# 第14回　PyTorch 入門
***
> **前提**: 第13回まで Scikit-learn で機械学習の基礎（Pipeline，正則化，学習曲線など）を学びました。第13回では決定木の `max_depth` による過学習も確認しました。ニューラルネットワークでも epoch 数や層の深さで同様のことが起きます。第15回以降で損失曲線を描いて確認します。
>
> ここから **PyTorch** を使い，深層学習フレームワークの基礎（Tensor，DataLoader，MNIST）を学びます。第15回以降 q14→q15→q16→q17 と PyTorch で一本につながります。

## 目次
1. Tensor と自動微分
2. MNIST の読み込み
3. MNIST の可視化
4. DataLoader の理解

---

## この回で学ぶこと

### なぜ PyTorch を使うのか

これまで使ってきた scikit-learn は「既成のモデルを使う」ためのライブラリだ。PyTorch は「自分でニューラルネットワークを設計・実装する」ためのフレームワークで，現在の深層学習研究の標準となっている。

| | scikit-learn | PyTorch |
|---|---|---|
| 対象 | 機械学習（線形回帰〜XGBoost） | 深層学習（NN, CNN, Transformer） |
| 柔軟性 | 低（既成モデルを使う） | 高（自由にモデルを設計） |
| 研究での利用 | 特徴量エンジニアリング，ベースライン | 最先端モデルの実装 |

卒業研究で「画像認識」「自然言語処理」「時系列予測」などを扱うなら，PyTorch は必須のスキルだ。

### Tensor とは

Tensor は PyTorch の基本データ構造で，NumPy の `ndarray` に似ているが，以下の点が異なる：
- **GPU で計算できる**（`.to("cuda")`）：大規模モデルの学習を劇的に高速化
- **自動微分（Autograd）** をサポート：ニューラルネットワークの学習に必要な勾配を自動計算

### 自動微分（Autograd）の重要性

ニューラルネットワークの学習には，損失を各パラメータで微分する「誤差逆伝播法（Backpropagation）」が必要だ。これを手計算すると非常に複雑になるが，PyTorch の `requires_grad=True` を使えば自動的に計算できる。

### ミニバッチ学習と DataLoader

全データを一度に学習する「バッチ学習」は：
- メモリに収まらない大規模データでは不可能
- 勾配の計算が遅い

`DataLoader` は大きなデータを小さな「ミニバッチ」に分割して，少しずつ学習できるようにする。`shuffle=True` にすることで，毎エポック異なる順序でデータが供給され，モデルが特定の順序に依存しない学習ができる。

### GPU と CPU の使い分け

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

- **GPU**: 深層学習の行列演算を並列処理で高速化（Colab の無料 GPU が使える）
- **CPU**: GPU がない環境でも動作する（今回のような小規模問題では十分）

データとモデルを同じデバイス（GPU か CPU）に置く必要がある。`.to(device)` でデバイスを指定する。

In [ ]:
%pip install -q torch torchvision


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA_ROOT = "./data"  # MNIST は torchvision が自動ダウンロード


## 問題1　Tensor と自動微分
***

### Tensor の基本操作

NumPy の配列に似ているが，型には注意が必要だ：
- `torch.tensor([1, 2, 3])` → デフォルトは int64
- `torch.tensor([1.0, 2.0, 3.0])` → float32（NN で使う型）
- `x.dtype` で型を確認できる

### 自動微分の仕組み

`requires_grad=True` を設定すると，その Tensor を使った計算履歴（計算グラフ）が記録される：

```
x = 5.0 (requires_grad=True)
y = 2x² + 3

y.backward() を呼ぶと：
  dy/dx = 4x を計算 → x=5 なので dy/dx = 20.0
```

これがニューラルネットワークの学習の本質だ：
```
損失L を各重みw で偏微分 → dL/dw を計算（これが勾配）
→ w = w - lr × dL/dw（勾配降下法で重みを更新）
```

PyTorch はこの「勾配の計算」を自動化している。

### 課題

PyTorch の **Tensor** の基礎と **自動微分** を確認してください。

1. `torch.tensor([1.0, 2.0, 3.0])` を作成し，`shape` と `dtype` を出力
2. `x = torch.tensor(5.0, requires_grad=True)` として `y = 2.0 * x ** 2 + 3.0` を計算
3. `y.backward()` の後，`x.grad.item()` で `dy/dx` を出力（正解: 20.0）

> **確認**: `y = 2x² + 3` を微分すると `dy/dx = 4x`，x=5 を代入すると 4×5 = 20.0 になるはずだ。

#### Hints
- `requires_grad=True` を付けた Tensor は、それを使った計算の履歴が自動的に記録される
- `y.backward()` を呼ぶと、`y` から辿れるすべての `requires_grad=True` な Tensor の勾配が計算される
- 計算された勾配は Tensor の `.grad` 属性に格納される。`.item()` で Python の数値として取り出せる

In [ ]:
# Tensor と自動微分
# ここにあなたのコードを書いてください


## 問題2　MNIST データセットの読み込み
***

### MNIST とは

MNIST（Modified National Institute of Standards and Technology）は，手書き数字（0〜9）の画像データセットだ：
- 訓練データ：60,000枚
- テストデータ：10,000枚
- 画像サイズ：28×28 ピクセル（グレースケール）
- クラス数：10（数字0〜9）

機械学習の "Hello World" として広く使われており，新しいモデルの動作確認に最適だ。

### `transforms.ToTensor()` が行うこと

`PIL.Image` 形式の画像（ピクセル値 0〜255, uint8）を PyTorch Tensor（値 0.0〜1.0, float32）に変換する：

```
原画像: shape (28, 28), dtype uint8, 値 0〜255
変換後: shape (1, 28, 28), dtype float32, 値 0.0〜1.0

【変換の意味】
・チャネル次元が追加（グレースケールなので 1 チャネル）
・値を 255 で割って正規化（NN の学習を安定化させる）
```

### `batch_size=64` の選び方

バッチサイズはハイパーパラメータの一つ：
- 大きいバッチ（256, 512）：GPU を効率よく使える，勾配が安定，メモリ消費大
- 小さいバッチ（32, 64）：メモリ消費小，ノイズが多く局所最適解から抜けやすいことも
- **64や128が多くの場面でバランスが良い**

### 課題

`torchvision.datasets.MNIST` で訓練データを読み込み，`DataLoader(batch_size=64, shuffle=True)` を作成してください。

データセットのサンプル数（`len(dataset)`）と，データセットの1件目の情報（画像の shape，ラベル）を出力してください。

#### Hints
- `datasets.MNIST` の `train=True` で訓練データ、`train=False` でテストデータを取得できる
- `transform` 引数に `transforms.ToTensor()` を渡すと、読み込み時に自動的に Tensor へ変換される
- `DataLoader` にデータセットを渡す際、`batch_size` と `shuffle` を設定する（訓練時は `shuffle=True` が推奨）
- データセットのインデックスアクセス（`dataset[0]`）で `(画像Tensor, ラベル)` のタプルが得られる

In [ ]:
# MNIST DataLoader
# ここにあなたのコードを書いてください


## 問題3　MNIST の可視化
***

### なぜデータを可視化するのか

「データを見る」ことは機械学習の最も基本的なステップだ。以下を確認することが重要：
1. データが正しく読み込めているか（画像が正しく表示されるか）
2. ラベルと画像が対応しているか
3. データの品質（ぼやけている，傾いている，など）

### `img.squeeze()` の意味

`ToTensor()` 適用後の画像は `(1, 28, 28)`（チャネル×高さ×幅）の shape を持つ。`plt.imshow()` は `(28, 28)` の2次元配列を期待するため，`squeeze()` でチャネル次元（サイズ1の次元）を除去する：

```
(1, 28, 28) → squeeze() → (28, 28)
```

### ピクセル値の確認

`ToTensor()` 適用後のピクセル値は 0.0〜1.0 の範囲になる。これは元の 0〜255 を255で割った値だ。NN への入力は常にこの範囲で行う（大きい値のまま入力すると学習が不安定になる）。

### 課題

MNIST の画像を **9枚** グリッド表示してください（3×3）。各画像の上に正解ラベルをタイトルとして付けてください。

また，表示した1枚について，`ToTensor()` 適用後のピクセル値の**最小値・最大値**を出力してください。

> **確認**: 画像とラベルが正しく対応しているか目視で確認しよう。

#### Hints
- `plt.subplots(3, 3)` で 3×3 のグリッドを作り、`axes.flat` でイテレートすると9枚分のループが書きやすい
- 画像 Tensor は `(1, 28, 28)` の形状なので、`imshow` に渡す前に `squeeze()` で余分な次元を除去する
- `ax.set_title` でラベルをタイトルとして設定し、`ax.axis("off")` で軸を非表示にするとすっきりする
- Tensor のピクセル値の最小・最大は `.min()` / `.max()` で取得できる（返り値は Tensor なので `.item()` で数値に変換）

In [ ]:
# MNIST 可視化
# ここにあなたのコードを書いてください


## 問題4　DataLoader とミニバッチの理解
***

### バッチのデータ構造

DataLoader が返す1バッチのデータ構造を理解することは，第15回以降の実装に直結する：

```
images: shape (64, 1, 28, 28)
  64 = バッチサイズ
   1 = チャネル数（グレースケール）
  28 = 画像の高さ（ピクセル）
  28 = 画像の幅（ピクセル）

labels: shape (64,)
  64個のラベル（0〜9の整数）
```

### Flatten（平坦化）の必要性

全結合層（Fully Connected Layer, `nn.Linear`）は**1次元ベクトル**を入力として受け取る。`(1, 28, 28)` の3次元テンソルをそのまま入れることはできないので，`(784,)` の1次元ベクトルに変形（flatten）する必要がある：

```
28 × 28 = 784 → これが全結合層への入力次元数
```

`view(-1, 784)` の `-1` は「残りの次元を自動計算する」という意味で，バッチサイズを保ったまま flatten できる：
```
(64, 1, 28, 28) → view(-1, 784) → (64, 784)
```

> **第15回との接続**: 第15回では `SimpleMLP` の `forward` メソッドで `x = x.view(-1, 784)` を最初に書く。これがこの flatten 操作だ。

### 課題

`train_loader` から **1バッチ** 取り出し，以下を出力してください。

1. `images.shape`（バッチサイズ，チャネル，高さ，幅）
2. `labels.shape` と最初の5件のラベル値
3. flatten 前 `(N, 1, 28, 28)` と flatten 後 `(N, 784)` の shape 比較

#### Hints
- `iter(loader)` でイテレータを作り、`next()` を呼ぶと1バッチ分のデータが取り出せる
- バッチの `images` の shape は `(バッチサイズ, チャネル, 高さ, 幅)` の4次元
- `view(-1, 784)` で flatten できる。`-1` はバッチサイズを自動計算することを意味する（28 × 28 = 784）

In [ ]:
# DataLoader の理解
# ここにあなたのコードを書いてください
